In [ ]:
import os
import json
import subprocess
import shutil
import requests
import base64
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
ELEVENLABS_API_KEY = "your_elevenlabs_api_key_here"
VOICE_ID           = "JBFqnCBsd6RMkjVDRZzb"
MODEL_ID           = "eleven_turbo_v2_5"

ORIGINAL_VIDEO     = "video.mp4"
AUDIO_DIR          = "audio_clips"
TEMP_DIR           = "temp_clips"
OUTPUT_VIDEO       = "final_demo.mp4"
AUDIO_META_PATH    = os.path.join(AUDIO_DIR, "audio_meta.json")
WORD_TIMES_PATH    = os.path.join(AUDIO_DIR, "word_timestamps.json")

FONT_NAME          = "Arial"
FONT_SIZE          = 52
NORMAL_COLOR       = "&H00FFFFFF"
HIGHLIGHT_COLOR    = "&H0000FFFF"
OUTLINE_COLOR      = "&H00000000"
BACK_COLOR         = "&H80000000"
MARGIN_V           = 40

PROBE_WORKERS      = 16   # parallel ffprobe — lightweight, safe to go high
ELEVEN_WORKERS     = 3    # ElevenLabs parallel — keep ≤3 on free tier

os.makedirs(TEMP_DIR, exist_ok=True)

# ─────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────
def run(cmd, label="", silent=False):
    if not silent:
        print(f"▶ {label}")
    result = subprocess.run(
        cmd,
        stdout=subprocess.DEVNULL,    # faster — skip capturing unused stdout
        stderr=subprocess.PIPE
    )
    if result.returncode != 0:
        print(f"❌ Error ({label}):\n{result.stderr.decode()[-600:]}")
        raise RuntimeError(label)

def get_duration(path):
    """Single ffprobe call — fast, no shell"""
    r = subprocess.run([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        path
    ], capture_output=True, text=True)
    try:
        return float(r.stdout.strip())
    except:
        return 0.0

def get_durations_parallel(paths, max_workers=PROBE_WORKERS):
    """Probe multiple files simultaneously — returns {path: duration}"""
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(get_duration, p): p for p in paths}
        for f in as_completed(futures):
            results[futures[f]] = f.result()
    return results

def seconds_to_ass(s):
    cs = int((s % 1) * 100)
    ss = int(s) % 60
    m  = int(s // 60) % 60
    h  = int(s // 3600)
    return f"{h}:{m:02d}:{ss:02d}.{cs:02d}"

# ─────────────────────────────────────────
# STEP 1: LOAD META + PARALLEL PROBE ALL FILES
# Probe video + all audio clips in one shot
# ─────────────────────────────────────────
t0 = time.time()

with open(AUDIO_META_PATH) as f:
    audio_meta = json.load(f)

# Collect all paths to probe at once
all_audio_paths = [
    os.path.join(AUDIO_DIR, e["audio"])
    for e in audio_meta
]
all_paths = [ORIGINAL_VIDEO] + all_audio_paths

print(f"⚡ Probing {len(all_paths)} files in parallel...")
durations = get_durations_parallel(all_paths)

video_duration = durations[ORIGINAL_VIDEO]
print(f"📹 Original video: {round(video_duration, 2)}s  "
      f"(probed in {round(time.time()-t0, 2)}s)")

# Build valid clips using pre-probed durations
valid_clips     = []
total_narration = 0.0

for entry in audio_meta:
    audio_path = os.path.join(AUDIO_DIR, entry["audio"])
    dur        = durations.get(audio_path, 0.0)
    if not os.path.exists(audio_path) or dur == 0.0:
        print(f"  ⚠️  Skipping: {audio_path}")
        continue
    valid_clips.append((entry, audio_path, dur))
    total_narration += dur

# Assign sequential timestamps
audio_entries   = []
cumulative_time = 0.0

for entry, audio_path, dur in valid_clips:
    audio_entries.append({
        "audio_path":  audio_path,
        "timestamp_s": round(cumulative_time, 4),
        "end_s":       round(cumulative_time + dur, 4),
        "duration":    dur,
        "narration":   entry.get("narration", ""),
        "position":    entry.get("position", "middle"),
    })
    cumulative_time += dur

master_duration = cumulative_time
slow_factor     = master_duration / video_duration if master_duration > video_duration else 1.0
need_stretch    = master_duration > video_duration

print(f"⏱️  Master duration: {round(master_duration, 2)}s  "
      f"| Slow factor: {round(slow_factor, 2)}x")

# ─────────────────────────────────────────
# STEP 2: FETCH WORD TIMESTAMPS
# Cached → instant. Otherwise parallel ElevenLabs calls.
# ─────────────────────────────────────────
def get_word_timestamps(text, narration_start_s, retries=3):
    url     = f"https://api.elevenlabs.io/v1/text-to-speech/{VOICE_ID}/with-timestamps"
    headers = {"xi-api-key": ELEVENLABS_API_KEY, "Content-Type": "application/json"}
    payload = {
        "text": text, "model_id": MODEL_ID,
        "voice_settings": {
            "stability": 0.5, "similarity_boost": 0.75,
            "style": 0.3, "use_speaker_boost": True
        }
    }

    for attempt in range(retries):
        try:
            r = requests.post(url, headers=headers, json=payload, timeout=30)
            if r.status_code == 200:
                break
            elif r.status_code == 429:
                time.sleep(2 ** attempt)
                continue
            else:
                return []
        except Exception as e:
            time.sleep(2 ** attempt)
    else:
        return []

    data        = r.json()
    alignment   = data.get("alignment", {})
    chars       = alignment.get("characters", [])
    char_starts = alignment.get("character_start_times_seconds", [])
    char_ends   = alignment.get("character_end_times_seconds", [])

    if not chars:
        return []

    # Character → word timestamps
    words, cur_word, word_start = [], "", None

    for i, (ch, cs, ce) in enumerate(zip(chars, char_starts, char_ends)):
        is_last = i == len(chars) - 1
        if ch == " " or is_last:
            if is_last and ch != " ":
                cur_word += ch
                ce = char_ends[i]
            elif i > 0:
                ce = char_ends[i - 1]
            if cur_word.strip() and word_start is not None:
                words.append({
                    "word":    cur_word.strip(),
                    "start_s": round(narration_start_s + word_start, 4),
                    "end_s":   round(narration_start_s + ce, 4),
                })
            cur_word, word_start = "", None
        else:
            if word_start is None:
                word_start = cs
            cur_word += ch

    return words

def fetch_words_worker(args):
    idx, ae = args
    words = get_word_timestamps(ae["narration"], ae["timestamp_s"])
    return idx, words

if os.path.exists(WORD_TIMES_PATH):
    print(f"\n⚡ Loading cached word timestamps...")
    with open(WORD_TIMES_PATH) as f:
        all_word_timestamps = json.load(f)
    print(f"✅ {len(all_word_timestamps)} words loaded from cache")
else:
    print(f"\n⚡ Fetching word timestamps ({len(audio_entries)} clips, "
          f"workers={ELEVEN_WORKERS})...")

    t1      = time.time()
    results = [None] * len(audio_entries)

    with ThreadPoolExecutor(max_workers=ELEVEN_WORKERS) as ex:
        futures = {
            ex.submit(fetch_words_worker, (i, ae)): i
            for i, ae in enumerate(audio_entries)
        }
        done = 0
        for future in as_completed(futures):
            idx, words = future.result()
            results[idx] = words or []
            done += 1
            print(f"  [{done}/{len(audio_entries)}] ✅ {len(words or [])} words")

    all_word_timestamps = [w for r in results if r for w in r]

    with open(WORD_TIMES_PATH, "w") as f:
        json.dump(all_word_timestamps, f, indent=2)

    print(f"✅ {len(all_word_timestamps)} words fetched in "
          f"{round(time.time()-t1, 2)}s")

# ─────────────────────────────────────────
# STEP 3: BUILD ASS + STRETCH VIDEO + BUILD AUDIO — ALL IN PARALLEL
# These 3 are completely independent → run simultaneously
# ─────────────────────────────────────────
stretched_video = os.path.join(TEMP_DIR, "stretched.mp4")
narration_track = os.path.join(TEMP_DIR, "narration.aac")
ass_path        = os.path.join(TEMP_DIR, "karaoke.ass")

def build_ass():
    print(f"🎨 [PARALLEL] Building karaoke ASS...")

    header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: 1920
PlayResY: 1080
Collisions: Normal

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Karaoke,{FONT_NAME},{FONT_SIZE},{NORMAL_COLOR},{HIGHLIGHT_COLOR},{OUTLINE_COLOR},{BACK_COLOR},1,0,0,0,100,100,0,0,3,2,0,2,20,20,{MARGIN_V},1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    events   = []
    word_idx = 0

    for ae in audio_entries:
        line_words = []
        for _ in ae["narration"].split():
            if word_idx < len(all_word_timestamps):
                line_words.append(all_word_timestamps[word_idx])
                word_idx += 1

        if not line_words:
            continue

        # Split into chunks of 8 words
        chunks = [line_words[i:i+8] for i in range(0, len(line_words), 8)]

        for chunk in chunks:
            start    = chunk[0]["start_s"]
            end      = chunk[-1]["end_s"]
            ktext    = ""
            for w in chunk:
                dur_cs  = max(int((w["end_s"] - w["start_s"]) * 100), 5)
                ktext  += f"{{\\k{dur_cs}}}{w['word']} "
            events.append(
                f"Dialogue: 0,{seconds_to_ass(start)},"
                f"{seconds_to_ass(end)},Karaoke,,0,0,0,,{ktext.strip()}"
            )

    with open(ass_path, "w", encoding="utf-8") as f:
        f.write(header + "\n".join(events))

    print(f"  ✅ ASS: {len(events)} blocks")
    return ass_path

def do_stretch():
    if need_stretch:
        print(f"🎞️  [PARALLEL] Stretching video {round(slow_factor,2)}x...")
        run([
            "ffmpeg", "-y",
            "-i", ORIGINAL_VIDEO,
            "-vf", f"setpts={slow_factor}*PTS",
            "-an", "-c:v", "libx264",
            "-preset", "fast", "-crf", "18",
            stretched_video
        ], label="Stretching", silent=True)
        print(f"  ✅ Stretched: {round(get_duration(stretched_video),2)}s")
        return stretched_video

🔧 Loading audio metadata...
📹 Original video: 88.86s
⏱️  Master duration: 178.56s

⚡ Loading cached word timestamps...
✅ Total words with timestamps: 393

🎨 Generating karaoke ASS subtitles...
✅ ASS file: 56 caption blocks

⚡ Stretching video + building audio in parallel...

🎞️  [PARALLEL] Stretching video 2.01x...
▶ Stretching video

🎚️  [PARALLEL] Building narration audio...
▶ Building narration
✅ Video: 178.34s
✅ Audio: 178.88s

🎬 Final render with karaoke captions...
▶ Rendering with karaoke captions
🧹 Cleaned up

────────────────────────────────────────────────────────────
✅ FINAL VIDEO READY
────────────────────────────────────────────────────────────
Output:          final_demo.mp4
Duration:        178.56s (2.98 min)
Caption style:   Karaoke — highlighted word ✅
Words tracked:   393
Size:            37.01 MB
────────────────────────────────────────────────────────────
